# 11. Searching Issues with JQL in Python

`jira.search_issues()` runs any **JQL** query (topic 05) and returns the matching issues. Anything you can find in the Jira search page, a script can find too.

> **Before you run this notebook:** create an API token (topic 08) and set `JIRA_SERVER`, `JIRA_EMAIL` and `JIRA_API_TOKEN` as environment variables, or in Colab's **Secrets** panel. The outputs shown were recorded on a practice Jira Cloud site with a project called **myproject** (key `MYP`). Your issue keys and data will differ.

In [1]:
import os
from jira import JIRA

jira = JIRA(
    server=os.environ["JIRA_SERVER"],                 # https://your-domain.atlassian.net
    basic_auth=(os.environ["JIRA_EMAIL"], os.environ["JIRA_API_TOKEN"]),
)

## A First Search

In [2]:
issue_in_project = jira.search_issues('project = MYP and issuetype = Task and status in ("In Progress", "To Do", "Resolved")')
print(len(issue_in_project))

# loop through the issues and print the key (ticket number) and summary
for issue in issue_in_project:
    print(f"Ticket: {issue.key}, Summary: {issue.fields.summary}")

5
Ticket: MYP-11, Summary: sample task
Ticket: MYP-10, Summary: Research Task
Ticket: MYP-9, Summary: Enhancement Request
Ticket: MYP-2, Summary: Sample Task 1
Ticket: MYP-1, Summary: Access to CloudOps cloud


- Put the JQL in single quotes so the double quotes inside it (`"In Progress"`) don't end the string.
- Results come back newest first by default. Add `ORDER BY created ASC` to reverse it.

### Getting More Than 50 Results

`search_issues()` returns up to **50** issues unless you ask for more:

```python
issues = jira.search_issues("project = MYP", maxResults=100)     # up to 100
issues = jira.search_issues("project = MYP", maxResults=False)   # every match, fetched in batches
```

## Task: A Ticket Report

Print a readable report for one ticket: who created it and when, plus its summary, description, reporter and assignee.

In [3]:
ticket = "MYP-8"
issue = jira.issue(ticket)

# extract the required details
reporter = issue.fields.reporter.displayName
assignee = issue.fields.assignee.displayName if issue.fields.assignee else "Unassigned"
created_date = issue.fields.created[:10]
summary = issue.fields.summary
description = issue.fields.description if issue.fields.description else "No description"

# format and print the output
output = f"""
TASK:
{reporter} ticket created --> {created_date}

Ticket details are as below:

Ticket Number: {ticket}

Ticket Summary: {summary}

Ticket Description: {description}

Ticket Reporter: {reporter}

Ticket Assignee: {assignee}
"""

print(output)

TASK:
Ahmad Hassan ticket created --> 2025-02-08

Ticket details are as below:

Ticket Number: MYP-8

Ticket Summary: Critical Bug

Ticket Description: I am unable to login the portal

Ticket Reporter: Ahmad Hassan

Ticket Assignee: Ahmad Hassan


> `issue.fields.assignee` is `None` for unassigned issues, and calling `.displayName` on `None` crashes. The `if ... else "Unassigned"` guard prevents that. Always guard optional fields this way.

## Task: Every Ticket One Person Created on One Day

Combine the report with a JQL search:

In [4]:
# tickets created by Ahmad Hassan on 2025-02-08
jql_query = 'created >= "2025-02-08 00:00" AND created <= "2025-02-08 23:59" AND reporter = "Ahmad Hassan"'

issues = jira.search_issues(jql_query, maxResults=100)
print(f"Total Tickets Created by Ahmad Hassan on 2025-02-08: {len(issues)}\n")

for issue in issues:
    reporter = issue.fields.reporter.displayName
    assignee = issue.fields.assignee.displayName if issue.fields.assignee else "Unassigned"
    created_date = issue.fields.created[:10]  # YYYY-MM-DD
    summary = issue.fields.summary
    description = issue.fields.description if issue.fields.description else "No description"

    output = f"""
TASK:
{reporter} ticket created --> {created_date}

Ticket details are as below:

Ticket Number: {issue.key}

Ticket Summary: {summary}

Ticket Description: {description}

Ticket Reporter: {reporter}

Ticket Assignee: {assignee}
"""
    print(output)

Total Tickets Created by Ahmad Hassan on 2025-02-08: 10


TASK:
Ahmad Hassan ticket created --> 2025-02-08

Ticket details are as below:

Ticket Number: MYP-10

Ticket Summary: Research Task

Ticket Description: No description

Ticket Reporter: Ahmad Hassan

Ticket Assignee: Ahmad Hassan


TASK:
Ahmad Hassan ticket created --> 2025-02-08

Ticket details are as below:

Ticket Number: MYP-9

Ticket Summary: Enhancement Request

Ticket Description: No description

Ticket Reporter: Ahmad Hassan

Ticket Assignee: Unassigned


TASK:
Ahmad Hassan ticket created --> 2025-02-08

Ticket details are as below:

Ticket Number: MYP-8

Ticket Summary: Critical Bug

Ticket Description: I am unable to login the portal

Ticket Reporter: Ahmad Hassan

Ticket Assignee: Ahmad Hassan


TASK:
Ahmad Hassan ticket created --> 2025-02-08

Ticket details are as below:

Ticket Number: MYP-7

Ticket Summary: Sample Story 2

Ticket Description: No description

Ticket Reporter: Ahmad Hassan

Ticket Assignee: Unassig

> Names with spaces work in JQL with quotes, but **account IDs** are safer in long-lived scripts: `reporter = "<account-id>"`. Display names can change, IDs never do.

## A Compact Table Instead

For many tickets, one line each is easier to read:

In [5]:
issues = jira.search_issues("project = MYP ORDER BY created DESC", maxResults=20)

print(f"{'Key':<8} {'Type':<7} {'Status':<12} {'Assignee':<14} Summary")
for issue in issues:
    f = issue.fields
    assignee = f.assignee.displayName if f.assignee else "Unassigned"
    print(f"{issue.key:<8} {f.issuetype.name:<7} {f.status.name:<12} {assignee:<14} {f.summary}")

## Practice: Run the JQL Tasks from Python

Topic 05 ended with four JQL tasks. Run them from Python and print how many issues each finds.

**Solution:**

In [6]:
queries = {
    "Highest priority IIP issues from Nov 2022": "project = IIP AND created >= 2022-11-01 AND created <= 2022-11-30 AND priority = Highest ORDER BY created DESC",
    "Started but no description": 'status != "To Do" AND description is EMPTY',
    "Unassigned for more than a day": "assignee is EMPTY AND created < -1d",
    "New unassigned MYP tasks (3 hours)": "project = MYP AND issuetype = Task AND created >= -3h AND assignee is EMPTY AND resolution = Unresolved",
}

for label, jql in queries.items():
    found = jira.search_issues(jql, maxResults=False)
    print(f"{label}: {len(found)}")